#### Load dataset

In [1]:
from dataset_simulation import get_dataloader, Simulation_Dataset
dataset = Simulation_Dataset(data_length=100, scenario="1-1", load_scenario_map=True, zero_based_position=True)

In [13]:
trainset, validset, testset = get_dataloader(data_length=100, seed=1,batch_size=32, scenario="1-1", load_scenario_map=True, zero_based_position=True)

Generated 3-channel map shape: (100, 840, 3) (Scale: 10x)
Generated 3-channel map shape: (100, 840, 3) (Scale: 10x)
Generated 3-channel map shape: (100, 840, 3) (Scale: 10x)


#### Load Model

In [4]:
import yaml
import json
from main_model import CSDI_SimulationScenmap
path = "config/base_scenmap.yaml"
with open(path, "r") as f:
    config = yaml.safe_load(f)
config["model"]["is_unconditional"] = False
config["model"]["test_missing_ratio"] = 0.1
print(json.dumps(config, indent=4))
model = CSDI_SimulationScenmap(config, "cpu")

{
    "train": {
        "epochs": 200,
        "batch_size": 16,
        "lr": 0.001,
        "itr_per_epoch": 100000000.0
    },
    "diffusion": {
        "layers": 4,
        "channels": 64,
        "nheads": 8,
        "diffusion_embedding_dim": 128,
        "beta_start": 0.0001,
        "beta_end": 0.5,
        "num_steps": 50,
        "schedule": "quad",
        "is_linear": false
    },
    "model": {
        "is_unconditional": false,
        "timeemb": 128,
        "featureemb": 16,
        "scenmapemb": 256,
        "target_strategy": "random",
        "test_missing_ratio": 0.1
    }
}


/home/dsv/qida0163/anaconda3/envs/csdi/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [14]:
for data_batch in trainset:
    batch = data_batch
    break

In [7]:
batch.keys()

dict_keys(['observed_data', 'observed_mask', 'gt_mask', 'timepoints', 'person_ids', 'scen_map'])

In [18]:
(
    observed_data,
    observed_mask,
    observed_tp,
    gt_mask,
    for_pattern_mask,
    _,
    scenmap,
) = model.process_data(batch)

In [19]:
scenmap.shape

torch.Size([32, 3, 100, 840])

In [26]:
cond_mask = model.get_randmask(observed_mask)
side_info = model.get_side_info(observed_tp, cond_mask, scenmap)

torch.Size([32, 256, 2, 100])
side_info: torch.Size([32, 145, 2, 100])
side_info after: torch.Size([32, 401, 2, 100])


In [28]:
loss = model.calc_loss(
    observed_data, cond_mask, observed_mask, side_info, 1
)

In [29]:
loss

tensor(0.9872, grad_fn=<DivBackward0>)